<a href="https://colab.research.google.com/github/fredoomnaxcj-creator/UTS_BIG-DATA_Aldi-Nurhadi/blob/main/UTS_BIG-DATA_ALDI-NURHADI_14022300014.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'app.bpjs.mobile',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

### Analisis Sentimen Ulasan

Selanjutnya, kita akan melakukan analisis sentimen pada ulasan yang telah dikumpulkan menggunakan model RoBERTa berbahasa Indonesia (`w11wo/indonesian-roberta-base-prdect-id`) dari Hugging Face.

In [ ]:
pip install transformers torch

In [ ]:
from transformers import pipeline

# Load the sentiment analysis model
sentiment_analyzer = pipeline("sentiment-analysis", model="w11wo/indonesian-roberta-base-prdect-id")

# Function to get sentiment for a given text
def get_sentiment(text):
    if not text:
        return {'label': 'NEUTRAL', 'score': 0.0} # Handle empty content
    try:
        result = sentiment_analyzer(text)
        return result[0]
    except Exception as e:
        print(f"Error analyzing sentiment for text: {text[:50]}... Error: {e}")
        return {'label': 'ERROR', 'score': 0.0}

# Apply sentiment analysis to each review
for review_item in result:
    sentiment = get_sentiment(review_item['content'])
    review_item['sentiment_label'] = sentiment['label']
    review_item['sentiment_score'] = sentiment['score']

print("Analisis sentimen selesai.")

### Menampilkan Beberapa Hasil Analisis Sentimen

In [ ]:
import pandas as pd

# Convert the list of dictionaries to a pandas DataFrame for easier display
df_reviews = pd.DataFrame(result)

# Display the first 10 reviews with sentiment information
display(df_reviews[['userName', 'content', 'score', 'sentiment_label', 'sentiment_score']].head(10))


### Memperbarui File CSV dengan Hasil Sentimen

Sekarang, kita akan menyimpan kembali ulasan beserta hasil analisis sentimen ke dalam file CSV yang sama.

In [ ]:
import csv

# Define the new fieldnames including sentiment columns
fieldnames_with_sentiment = ['userName', 'score', 'at', 'content', 'sentiment_label', 'sentiment_score']

updated_filename = 'ulasan_google_play_dengan_sentimen.csv'

with open(updated_filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames_with_sentiment)
    writer.writeheader()
    for review_item in result:
        # Ensure all fields are present to avoid KeyError, using .get() with default values
        writer.writerow({
            'userName': review_item.get('userName', ''),
            'score': review_item.get('score', ''),
            'at': review_item.get('at', ''),
            'content': review_item.get('content', ''),
            'sentiment_label': review_item.get('sentiment_label', 'N/A'),
            'sentiment_score': review_item.get('sentiment_score', 0.0)
        })

print(f"Berhasil menyimpan {len(result)} ulasan beserta sentimen ke '{updated_filename}'")

### Visualisasi Distribusi Sentimen

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count the occurrences of each sentiment label
sentiment_counts = df_reviews['sentiment_label'].value_counts().reset_index()
sentiment_counts.columns = ['Sentiment', 'Count']

# Create the bar plot
plt.figure(figsize=(8, 6))
sns.barplot(x='Sentiment', y='Count', hue='Sentiment', data=sentiment_counts, palette='viridis', legend=False)
plt.title('Distribusi Sentimen Ulasan Google Play', fontsize=16)
plt.xlabel('Label Sentimen', fontsize=12)
plt.ylabel('Jumlah Ulasan', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()